# Uniform-split training-data amount sweep with per-fraction Keras-tuner search

This notebook combines the two existing data-amount sweeps:

- Like `ml_35_uniform_data_amount_sweep.ipynb`, **every split is drawn uniformly at random from across the whole parameter space** — no held-out geometry corner and no far-to-near extrapolation. The test and validation sets are a uniform random sample of the full dataset (`SPLIT_SEED = 42`, identical to ml_35/ml_36, so the held-out rows are the same), and each training fraction (10%, 20%, ..., 100%) is a nested uniform random subset of the remaining training pool.
- Like `ml_32_corner_far_to_near.ipynb`, each fraction gets an **independent Keras-tuner (BayesianOptimization) hyperparameter search** instead of copying a fixed reference architecture. For each fraction it runs two searches: a forward-surrogate search (`KERAS_TUNER_TRIALS_SURROGATE` trials, search space mirrors `ml_11_train_keras_surrogate.ipynb`) and an inverse+frozen-surrogate combined search (`KERAS_TUNER_TRIALS_INVERSE` trials, search space mirrors `ml_21_train_keras_surrogate_defined_loss.ipynb`).

Single seed (`SEEDS = (0,)`). It writes:

- `data_amount_sweep_uniform_keras_tuner_seed0.csv` (same hyperparameter column schema as the ml_32 CSV, so the fixed-HP companion notebook can read it the way ml_37 reads ml_32's)
- `data_amount_sweep_uniform_keras_tuner_seed0_summary.csv`
- `model/uniform_data_amount_sweep_keras_tuner/fraction_*pct_seed0_{surrogate,combined,inverse}.keras`

The companion notebook `ml_39_uniform_sweep_fixed_hp_predictions.ipynb` takes the per-fraction best hyperparameters from the CSV, retrains with them, and writes 20 per-sample predictions per training slice.

100% means 100% of the (uniformly sampled) non-held-out training pool. The uniformly sampled validation and test rows are always excluded from training.


In [1]:
from __future__ import annotations

import gc
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
# Use the async CUDA allocator to avoid fragmentation OOMs over the long
# multi-fraction training loop. Must be set before TensorFlow touches the GPU.
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.models import load_model


tf.keras.backend.set_floatx("float32")

# Let GPU memory grow on demand instead of pre-reserving it all, which also
# reduces fragmentation OOMs during the long run.
for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass


# This notebook can be run either from the transmon experiment folder or from the
# repo root. The block below tries both so local paths do not need hard-coding.
HERE = Path.cwd()
EXPERIMENT_RELATIVE = Path("experiments/model_predict_qubit_TransmonCross_Hamiltonian_params")

if (HERE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE
elif (HERE / EXPERIMENT_RELATIVE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE / EXPERIMENT_RELATIVE
else:
    raise FileNotFoundError(
        "Could not find the transmon-cross metadata file. "
        "Run this notebook from the repo root or from the transmon experiment folder."
    )

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from parameters_surrogate_defined_loss import (  # noqa: E402
    EPOCHS,
    KT_DIR,
    MODEL_DIR as PARAM_MODEL_DIR,
    SCALERS_DIR as PARAM_SCALERS_DIR,
    TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS,
)
from parameters_surrogate import (  # noqa: E402
    EPOCHS as SURROGATE_EPOCHS,
    TRAIN_BATCH_SIZE as SURROGATE_TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE as SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS as SURROGATE_TRAIN_LOSS,
)

METADATA_DIR = EXPERIMENT_DIR / "metadata"
METADATA_PATH = METADATA_DIR / "qubit-TransmonCross-Hamiltonian_params.json"
OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_uniform_keras_tuner_seed0.csv"
SUMMARY_OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_uniform_keras_tuner_seed0_summary.csv"

MODEL_DIR = Path(PARAM_MODEL_DIR)
SCALERS_DIR = Path(PARAM_SCALERS_DIR)
SWEEP_MODEL_DIR = MODEL_DIR / "uniform_data_amount_sweep_keras_tuner"
SWEEP_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Sweep all ten training-pool fractions (10% ... 100%).
FRACTIONS = tuple(round(0.1 * i, 2) for i in range(1, 11))
# Single tuner seed. The seed is baked into OUT_PATH, so update both together.
SEEDS = (0,)

# Uniform random split across the whole parameter space (no held-out corner).
# Same SPLIT_SEED as ml_35/ml_36, so the test/validation/training-pool split (and
# therefore the prediction targets downstream) match the uniform-sweep pipeline.
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15
SPLIT_SEED = 42

# Keras-tuner hyperparameter search per fraction. 50 surrogate + 50 inverse trials
# = 100 trials per fraction (1000 trials across the ten fractions).
# The surrogate search space mirrors ml_11; the inverse search space mirrors ml_21.
KERAS_TUNER_TRIALS_SURROGATE = 50
KERAS_TUNER_TRIALS_INVERSE = 50
KERAS_TUNER_EXECUTIONS_PER_TRIAL = 1
# Tolerate a few transient OOM/failed trials before a search gives up.
KERAS_TUNER_MAX_CONSEC_FAILURES = 8
# Resume (do not overwrite) so an interrupted long sweep can continue where it left off.
# Set to True to discard existing tuner trials and start the search from scratch.
KERAS_TUNER_OVERWRITE = False
TUNER_DIR = Path(KT_DIR) / "uniform_data_amount_sweep"
TUNER_DIR.mkdir(parents=True, exist_ok=True)

# Training-loop settings for the inverse (combined) model, shared with ml_21/ml_32.
INVERSE_EPOCHS = EPOCHS
INVERSE_BATCH_SIZE = TRAIN_BATCH_SIZE
INVERSE_EARLY_STOPPING_PATIENCE = TRAIN_EARLY_STOPPING_PATIENCE

# Run the training sweep. Set this to False if the CSV already exists and you
# only want to re-read the results.
RUN_SWEEP = True

EPS = 1e-12


In [2]:
# data loading

@dataclass
class Scaler:
    min_: np.ndarray
    max_: np.ndarray

    @property
    def range_(self) -> np.ndarray:
        return np.maximum(self.max_ - self.min_, EPS)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.min_) / self.range_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * self.range_ + self.min_


def parse_um(value: object) -> float:
    text = str(value).strip()
    for suffix in ("um", "µm", "μm"):
        if text.endswith(suffix):
            return float(text[: -len(suffix)])
    return float(text)


def load_arrays() -> tuple[np.ndarray, np.ndarray]:
    """Load Hamiltonian targets and geometry values from the SQuADDS metadata."""
    data = json.loads(METADATA_PATH.read_text())
    hamiltonian = []
    geometry = []

    for row in data:
        h = row["Hamiltonian_params"]
        opts = row["design"]["design_options"]
        readout = opts["connection_pads"]["readout"]

        hamiltonian.append(
            [
                float(h["qubit_frequency_GHz"]),
                float(h["anharmonicity_MHz"]),
            ]
        )
        geometry.append(
            [
                parse_um(readout["claw_length"]),
                parse_um(readout["ground_spacing"]),
                parse_um(opts["cross_length"]),
            ]
        )

    return np.asarray(hamiltonian, dtype=np.float64), np.asarray(geometry, dtype=np.float64)


def choose_uniform_split(
    n_rows: int,
    test_fraction: float,
    val_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Draw test/validation/training-pool indices uniformly at random.

    Unlike the corner notebook, no geometry corner is held out. The test and
    validation rows are a uniform random sample from across the whole parameter
    space, and the rest becomes the training pool.
    """
    n_test = int(np.ceil(test_fraction * n_rows))
    n_val = int(np.ceil(val_fraction * n_rows))

    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_rows)

    test_idx = perm[:n_test]
    val_idx = perm[n_test:n_test + n_val]
    train_pool_idx = perm[n_test + n_val:]

    return train_pool_idx, val_idx, test_idx


def uniform_training_order(train_pool_idx: np.ndarray, seed: int) -> np.ndarray:
    """
    Return a uniform random ordering (local indices) of the training pool.

    Taking the first ``n`` of this ordering gives nested uniform random subsets
    as the training fraction grows, the analogue of the corner notebook's
    far-to-near ordering but without any spatial structure.
    """
    rng = np.random.default_rng(seed + 1000)
    return rng.permutation(len(train_pool_idx))


def scaler_from_artifacts(
    values: np.ndarray,
    columns: list[str],
    path_patterns: list[str],
    label: str,
) -> tuple[Scaler, list[str]]:
    """Load per-column MinMaxScaler ranges when present, otherwise fit on metadata."""
    mins = []
    maxs = []
    sources = []

    for i, col in enumerate(columns):
        loaded = None
        source = None
        for pattern in path_patterns:
            candidate = SCALERS_DIR / pattern.format(col=col)
            if candidate.exists():
                loaded = joblib.load(candidate)
                source = str(candidate)
                break

        if loaded is not None:
            mins.append(float(np.asarray(loaded.data_min_).reshape(-1)[0]))
            maxs.append(float(np.asarray(loaded.data_max_).reshape(-1)[0]))
            sources.append(source)
        else:
            mins.append(float(np.min(values[:, i])))
            maxs.append(float(np.max(values[:, i])))
            sources.append(f"metadata fallback: {label}.{col}")

    return Scaler(np.asarray(mins), np.asarray(maxs)), sources


In [3]:
# building the split and scalers

h_raw, geom_raw_um = load_arrays()
geom_raw_si = geom_raw_um * 1e-6

HAMILTONIAN_COLUMN_NAMES = (METADATA_DIR / "X_names").read_text().splitlines()
QISKIT_PARAM_NAMES = np.load(METADATA_DIR / "y_columns.npy", allow_pickle=True).astype(str).tolist()

train_pool_idx, val_idx, test_idx = choose_uniform_split(
    len(geom_raw_um),
    TEST_FRACTION,
    VAL_FRACTION,
    SPLIT_SEED,
)

# Make sure the held-out validation/test rows never leak into the training pool.
assert len(np.intersect1d(train_pool_idx, val_idx)) == 0
assert len(np.intersect1d(train_pool_idx, test_idx)) == 0
assert len(np.intersect1d(val_idx, test_idx)) == 0

# Uniform random ordering of the training pool; nested subsets grow with fraction.
uniform_order = uniform_training_order(train_pool_idx, SPLIT_SEED)

# For the actual model inputs/outputs, use the saved scaler artifacts from the
# existing repo workflow. These are fit on the complete dataset, so the scaled
# [0, 1] range used by the range penalty is the total space spanned by the
# complete dataset. This also keeps the surrogate in its original scaled space.
h_model_scaler, h_scaler_sources = scaler_from_artifacts(
    h_raw,
    HAMILTONIAN_COLUMN_NAMES,
    ["scaler_X_{col}.save", "scaler_X_linear_{col}.save"],
    "Hamiltonian",
)
geom_inverse_scaler, geom_inverse_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_{col}_one_hot_encoding.save"],
    "inverse_qiskit",
)
geom_surrogate_scaler, geom_surrogate_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_linear_{col}.save", "scaler_y_{col}_one_hot_encoding.save"],
    "surrogate_qiskit",
)

h_model_scaled = h_model_scaler.transform(h_raw).astype("float32")
geom_inverse_scaled = geom_inverse_scaler.transform(geom_raw_si).astype("float32")
geom_surrogate_scaled = geom_surrogate_scaler.transform(geom_raw_si).astype("float32")

# Convert inverse output scaler space to surrogate input scaler space.
# surrogate_scaled = inverse_scaled * scale_a + scale_b
scale_a = (geom_inverse_scaler.range_ / geom_surrogate_scaler.range_).astype("float32")
scale_b = ((geom_inverse_scaler.min_ - geom_surrogate_scaler.min_) / geom_surrogate_scaler.range_).astype("float32")

print(f"Loaded {len(h_raw)} total samples")
print(f"Training pool: {len(train_pool_idx)}")
print(f"Validation (uniform): {len(val_idx)}")
print(f"Test (uniform): {len(test_idx)}")
print()
print("Split check passed:")
print("  train/val overlap:", len(np.intersect1d(train_pool_idx, val_idx)))
print("  train/test overlap:", len(np.intersect1d(train_pool_idx, test_idx)))
print("  val/test overlap:", len(np.intersect1d(val_idx, test_idx)))
print("  100% means all non-held-out training-pool samples, not all samples.")
print()
print("Using saved model-space scalers from the existing repo workflow.")
for name, lo, hi, source in zip(HAMILTONIAN_COLUMN_NAMES, h_model_scaler.min_, h_model_scaler.max_, h_scaler_sources):
    print(f"  Hamiltonian {name}: {lo:.6g} to {hi:.6g}  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_inverse_scaler.min_, geom_inverse_scaler.max_, geom_inverse_scaler_sources):
    print(f"  Inverse geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_surrogate_scaler.min_, geom_surrogate_scaler.max_, geom_surrogate_scaler_sources):
    print(f"  Surrogate geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")

if any(source.startswith("metadata fallback") for source in h_scaler_sources + geom_inverse_scaler_sources + geom_surrogate_scaler_sources):
    print()
    print("Note: at least one scaler artifact was not found, so metadata-derived min/max ranges were used for that column.")


Loaded 1934 total samples
Training pool: 1352
Validation (uniform): 291
Test (uniform): 291

Split check passed:
  train/val overlap: 0
  train/test overlap: 0
  val/test overlap: 0
  100% means all non-held-out training-pool samples, not all samples.

Using saved model-space scalers from the existing repo workflow.
  Hamiltonian qubit_frequency_GHz: 3.21853 to 7.12659  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_qubit_frequency_GHz.save)
  Hamiltonian anharmonicity_MHz: -525.818 to -88.9577  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_anharmonicity_MHz.save)
  Inverse geometry design_options.connection_pads.readout.claw_length: 7e-05 to 0.0004 SI units  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_y_design_options.connection_pads.readout.claw_length_one_hot_encoding.save)
  Inverse ge

In [4]:
# Shared model pieces: the scaled-space conversion between the inverse output and
# the surrogate input, the out-of-range penalty, and the frozen-surrogate loader.
# Identical to ml_32/ml_35/ml_37 so the saved models stay interchangeable.


class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault("trainable", False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = {"scale_a": list(np.asarray(scale_a, dtype=float)), "scale_b": list(np.asarray(scale_b, dtype=float))}

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config


def qiskit_range_penalty(y_true_dummy, y_pred):
    """Penalize inverse predictions outside the scaled [0, 1] training range."""
    below = tf.nn.relu(-y_pred)
    above = tf.nn.relu(y_pred - 1.0)
    return tf.reduce_mean(below ** 2 + above ** 2)


def load_frozen_surrogate(surrogate_model_path: Path):
    surrogate_model = load_model(surrogate_model_path, compile=False)
    surrogate_model.trainable = False
    for layer in surrogate_model.layers:
        layer.trainable = False
    return surrogate_model


In [5]:
# Keras-tuner hyperparameter search for the surrogate and the inverse+surrogate
# models. The surrogate search space mirrors ml_11_train_keras_surrogate.ipynb and
# the inverse search space mirrors ml_21_train_keras_surrogate_defined_loss.ipynb.

def make_surrogate_hypermodel(input_dim: int, output_dim: int):
    """Surrogate (geometry -> Hamiltonian) hypermodel, search space from ml_11."""
    def build(hp):
        tf.keras.backend.clear_session()
        gc.collect()

        n_layers = hp.Int("n_layers", min_value=1, max_value=1, default=1)
        neurons_per_layer = [hp.Int(f"neurons_{i}", min_value=32, max_value=1024, step=32) for i in range(n_layers)]
        dropout_rate = hp.Float("dropout_rate", 0.0, 0.45, step=0.05)
        l2_reg = hp.Float("l2_reg", 1e-5, 1e-2, sampling="LOG", default=1e-4)
        lr_initial = hp.Float("learning_rate", 3e-2, 1e-1, sampling="LOG", default=3e-2)
        use_batchnorm = hp.Boolean("use_batchnorm", default=True)

        model = Sequential(name="retrained_surrogate")
        model.add(Input(shape=(input_dim,), name="input1"))
        for i, n_units in enumerate(neurons_per_layer):
            model.add(Dense(n_units, name=f"fc{i}", kernel_initializer="he_normal",
                            kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
            if use_batchnorm:
                model.add(tf.keras.layers.BatchNormalization(name=f"bn{i}"))
            model.add(LeakyReLU(negative_slope=0.01, name=f"leaky_relu{i}"))
            model.add(Dropout(rate=dropout_rate, name=f"dropout{i}"))
        model.add(Dense(output_dim, name="output", kernel_initializer="he_normal"))
        model.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
                      loss=SURROGATE_TRAIN_LOSS, metrics=[SURROGATE_TRAIN_LOSS])
        return model
    return build


def make_inverse_hypermodel(surrogate_model_path: Path, h_dim: int, qiskit_dim: int):
    """Inverse+frozen-surrogate combined hypermodel, search space from ml_21."""
    def build(hp):
        tf.keras.backend.clear_session()
        gc.collect()

        surrogate_model = load_frozen_surrogate(surrogate_model_path)
        converter = ScalerConversionLayer(scale_a, scale_b, name="scaler_conversion")

        n_layers = hp.Int("n_layers", min_value=1, max_value=4, default=2)
        neurons_per_layer = [hp.Int(f"neurons_{i}", min_value=64, max_value=1024, step=64) for i in range(n_layers)]
        dropout_rate = hp.Float("dropout_rate", 0.0, 0.3, step=0.05)
        l2_reg = hp.Float("l2_reg", 1e-6, 1e-2, sampling="LOG", default=1e-6)
        lr_initial = hp.Float("learning_rate", 1e-3, 1e-1, sampling="LOG", default=1e-2)
        use_batchnorm = hp.Boolean("use_batchnorm", default=True)
        penalty_weight = hp.Float("penalty_weight", 0.01, 1.0, sampling="LOG", default=0.1)

        inverse_model = Sequential(name="inverse_model")
        inverse_model.add(Input(shape=(h_dim,), name="Hamiltonian_input"))
        for i, n_units in enumerate(neurons_per_layer):
            inverse_model.add(Dense(n_units, name=f"fc{i}", kernel_initializer="he_normal",
                                    kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
            if use_batchnorm:
                inverse_model.add(tf.keras.layers.BatchNormalization(name=f"bn{i}"))
            inverse_model.add(LeakyReLU(negative_slope=0.01, name=f"leaky_relu{i}"))
            inverse_model.add(Dropout(rate=dropout_rate, name=f"dropout{i}"))
        inverse_model.add(Dense(qiskit_dim, name="qiskit_output", kernel_initializer="he_normal"))

        combined_input = Input(shape=(h_dim,), name="combined_input")
        predicted_qiskit = inverse_model(combined_input)
        predicted_qiskit_converted = converter(predicted_qiskit)
        reconstructed_hamiltonian = surrogate_model(predicted_qiskit_converted)
        combined_model = Model(
            inputs=combined_input,
            outputs=[reconstructed_hamiltonian, predicted_qiskit],
            name="combined_model",
        )
        combined_model.compile(
            optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
            loss=[TRAIN_LOSS, qiskit_range_penalty],
            loss_weights=[1.0, penalty_weight],
        )
        return combined_model
    return build


def _hp_neurons(best_hp) -> list:
    n_layers = int(best_hp.get("n_layers"))
    return [int(best_hp.get(f"neurons_{i}")) for i in range(n_layers)]


def _best_trial_stats(tuner) -> tuple:
    """Return (best val_loss score, best epoch) for the tuner's best trial."""
    try:
        trial = tuner.oracle.get_best_trials(1)[0]
        score = float(trial.score) if trial.score is not None else float("nan")
        best_step = int(trial.best_step) if trial.best_step is not None else -1
    except Exception:
        score, best_step = float("nan"), -1
    return score, best_step


def tune_surrogate_for_subset(
    geom_subset_scaled: np.ndarray,
    h_subset_scaled: np.ndarray,
    geom_val_scaled: np.ndarray,
    h_val_scaled: np.ndarray,
    fraction: float,
    seed: int,
):
    pct = int(round(fraction * 100))
    tuner = kt.BayesianOptimization(
        make_surrogate_hypermodel(geom_subset_scaled.shape[1], h_subset_scaled.shape[1]),
        objective="val_loss",
        max_trials=KERAS_TUNER_TRIALS_SURROGATE,
        executions_per_trial=KERAS_TUNER_EXECUTIONS_PER_TRIAL,
        max_consecutive_failed_trials=KERAS_TUNER_MAX_CONSEC_FAILURES,
        seed=seed,
        directory=str(TUNER_DIR),
        project_name=f"surrogate_frac{pct:03d}_seed{seed}",
        overwrite=KERAS_TUNER_OVERWRITE,
    )
    early_stopping = EarlyStopping(
        monitor="val_loss", mode="min",
        patience=SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True, verbose=0,
    )
    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=max(10, SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE // 3),
        min_lr=1e-6, verbose=0,
    )
    tuner.search(
        np.asarray(geom_subset_scaled, dtype="float32"),
        np.asarray(h_subset_scaled, dtype="float32"),
        epochs=SURROGATE_EPOCHS,
        batch_size=SURROGATE_TRAIN_BATCH_SIZE,
        validation_data=(
            np.asarray(geom_val_scaled, dtype="float32"),
            np.asarray(h_val_scaled, dtype="float32"),
        ),
        callbacks=[early_stopping, reduce_lr],
        verbose=0,
    )
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]
    best_val_loss, best_step = _best_trial_stats(tuner)
    return best_model, best_hp, best_val_loss, best_step


def tune_inverse_for_subset(
    h_subset_scaled: np.ndarray,
    h_val_scaled: np.ndarray,
    fraction: float,
    seed: int,
    surrogate_model_path: Path,
):
    pct = int(round(fraction * 100))
    qiskit_dim = len(QISKIT_PARAM_NAMES)
    dummy_train = np.zeros((len(h_subset_scaled), qiskit_dim), dtype="float32")
    dummy_val = np.zeros((len(h_val_scaled), qiskit_dim), dtype="float32")

    tuner = kt.BayesianOptimization(
        make_inverse_hypermodel(surrogate_model_path, h_subset_scaled.shape[1], qiskit_dim),
        objective="val_loss",
        max_trials=KERAS_TUNER_TRIALS_INVERSE,
        executions_per_trial=KERAS_TUNER_EXECUTIONS_PER_TRIAL,
        max_consecutive_failed_trials=KERAS_TUNER_MAX_CONSEC_FAILURES,
        seed=seed,
        directory=str(TUNER_DIR),
        project_name=f"inverse_frac{pct:03d}_seed{seed}",
        overwrite=KERAS_TUNER_OVERWRITE,
    )
    early_stopping = EarlyStopping(
        monitor="val_loss", mode="min",
        patience=INVERSE_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True, verbose=0,
    )
    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=max(10, INVERSE_EARLY_STOPPING_PATIENCE // 3),
        min_lr=1e-6, verbose=0,
    )
    tuner.search(
        np.asarray(h_subset_scaled, dtype="float32"),
        [np.asarray(h_subset_scaled, dtype="float32"), dummy_train],
        epochs=INVERSE_EPOCHS,
        batch_size=INVERSE_BATCH_SIZE,
        validation_data=(
            np.asarray(h_val_scaled, dtype="float32"),
            [np.asarray(h_val_scaled, dtype="float32"), dummy_val],
        ),
        callbacks=[early_stopping, reduce_lr],
        verbose=0,
    )
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_combined = tuner.get_best_models(1)[0]
    best_inverse = best_combined.get_layer("inverse_model")
    best_val_loss, best_step = _best_trial_stats(tuner)
    return best_inverse, best_combined, best_hp, best_val_loss, best_step


def evaluate_percent_error(
    combined_model: Model,
    h_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict:
    h_pred_scaled, _ = combined_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)

    return {
        "omega_q_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def evaluate_surrogate_model(
    surrogate_model: Model,
    geom_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict:
    h_pred_scaled = surrogate_model.predict(np.asarray(geom_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)
    return {
        "omega_q_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def inverse_range_stats(inverse_model: Sequential, h_scaled_in: np.ndarray) -> dict:
    qiskit_scaled = inverse_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    below = np.maximum(-qiskit_scaled, 0.0)
    above = np.maximum(qiskit_scaled - 1.0, 0.0)
    violation = below + above
    return {
        "qiskit_scaled_min": float(np.min(qiskit_scaled)),
        "qiskit_scaled_max": float(np.max(qiskit_scaled)),
        "qiskit_range_violation_mean": float(np.mean(violation)),
        "qiskit_range_violation_max": float(np.max(violation)),
    }

In [6]:
# Per-fraction Keras-tuner search mode

print("Running a Keras-tuner (BayesianOptimization) search per fraction/seed.")
print(f"  surrogate trials per fraction: {KERAS_TUNER_TRIALS_SURROGATE} (search space mirrors ml_11)")
print(f"  inverse trials per fraction:   {KERAS_TUNER_TRIALS_INVERSE} (search space mirrors ml_21)")
print("Tuner working directory:", TUNER_DIR)
print("Sweep model output dir:", SWEEP_MODEL_DIR)


Running a Keras-tuner (BayesianOptimization) search per fraction/seed.
  surrogate trials per fraction: 50 (search space mirrors ml_11)
  inverse trials per fraction:   50 (search space mirrors ml_21)
Tuner working directory: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/kt_dir2/uniform_data_amount_sweep
Sweep model output dir: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/uniform_data_amount_sweep_keras_tuner


In [ ]:
# run sweep
def model_paths_for_fraction_seed(fraction: float, seed: int) -> tuple:
    pct = int(round(fraction * 100))
    stem = f"fraction_{pct:03d}pct_seed{seed}"
    return (
        SWEEP_MODEL_DIR / f"{stem}_surrogate.keras",
        SWEEP_MODEL_DIR / f"{stem}_combined.keras",
        SWEEP_MODEL_DIR / f"{stem}_inverse.keras",
    )


if RUN_SWEEP:
    print(f"Tuning a surrogate ({KERAS_TUNER_TRIALS_SURROGATE} trials) and inverse "
          f"({KERAS_TUNER_TRIALS_INVERSE} trials) per fraction.")
    print(f"Tuner working directory: {TUNER_DIR}")

    # Resume support: reload results already written in a previous run so we do
    # not repeat finished fractions or clobber their rows when the CSV is rewritten.
    if OUT_PATH.exists():
        rows = pd.read_csv(OUT_PATH).to_dict("records")
        done_keys = {(int(round(r["fraction"] * 100)), int(r["seed"])) for r in rows}
        print(f"Resuming from {OUT_PATH}: {len(rows)} existing rows for "
              f"fractions/seeds {sorted(done_keys)}")
    else:
        rows = []
        done_keys = set()
    n_train_pool = len(train_pool_idx)

    for fraction in FRACTIONS:
        n_subset = max(1, int(round(fraction * n_train_pool)))
        subset_local = uniform_order[:n_subset]
        subset_idx = train_pool_idx[subset_local]

        print(f"\n=== {fraction:.0%} of training pool ({n_subset} samples) ===")

        for seed in SEEDS:
            surrogate_model_path, combined_model_path, inverse_model_path = model_paths_for_fraction_seed(fraction, seed)

            # Skip only when the combined model is on disk AND a row already exists.
            # The combined model is saved after the inverse search succeeds, so a
            # half-finished fraction (surrogate done, inverse crashed) is re-run.
            if (int(round(fraction * 100)), int(seed)) in done_keys and combined_model_path.exists():
                print(f"  skipping {fraction:.0%} seed {seed} (already complete)")
                continue

            print(f"  [surrogate] tuning {KERAS_TUNER_TRIALS_SURROGATE} trials...")
            surrogate_model, surrogate_hp, surrogate_best_val_loss, surrogate_best_step = tune_surrogate_for_subset(
                geom_surrogate_scaled[subset_idx],
                h_model_scaled[subset_idx],
                geom_surrogate_scaled[val_idx],
                h_model_scaled[val_idx],
                fraction=fraction,
                seed=seed,
            )
            surrogate_model.save(surrogate_model_path)
            surrogate_hidden_units = _hp_neurons(surrogate_hp)
            print(f"    best surrogate units={surrogate_hidden_units} "
                  f"lr={float(surrogate_hp.get('learning_rate')):.4g} val_loss={surrogate_best_val_loss:.6g}")

            surrogate_train_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[subset_idx],
                h_raw[subset_idx],
            )
            surrogate_val_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[val_idx],
                h_raw[val_idx],
            )
            surrogate_test_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[test_idx],
                h_raw[test_idx],
            )

            del surrogate_model
            tf.keras.backend.clear_session()
            gc.collect()

            print(f"  [inverse] tuning {KERAS_TUNER_TRIALS_INVERSE} trials...")
            inverse_model, combined_model, inverse_hp, inverse_best_val_loss, inverse_best_step = tune_inverse_for_subset(
                h_model_scaled[subset_idx],
                h_model_scaled[val_idx],
                fraction=fraction,
                seed=seed,
                surrogate_model_path=surrogate_model_path,
            )
            combined_model.save(combined_model_path)
            inverse_model.save(inverse_model_path)
            inverse_hidden_units = _hp_neurons(inverse_hp)
            print(f"    best inverse units={inverse_hidden_units} "
                  f"lr={float(inverse_hp.get('learning_rate')):.4g} "
                  f"penalty={float(inverse_hp.get('penalty_weight')):.4g} val_loss={inverse_best_val_loss:.6g}")

            train_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[subset_idx],
                h_raw[subset_idx],
            )
            val_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[val_idx],
                h_raw[val_idx],
            )
            test_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[test_idx],
                h_raw[test_idx],
            )
            test_range_stats = inverse_range_stats(inverse_model, h_model_scaled[test_idx])

            rows.append(
                {
                    "selection_method": "uniform_random_split_nested_random_subsets_keras_tuner",
                    "split_seed": SPLIT_SEED,
                    "fraction": fraction,
                    "training_percent": fraction * 100.0,
                    "n_samples": n_subset,
                    "seed": seed,
                    "surrogate_training_mode": "keras_tuner_retrained_on_same_subset_as_inverse",
                    "saved_surrogate_model_path": str(surrogate_model_path),
                    "saved_combined_model_path": str(combined_model_path),
                    "saved_inverse_model_path": str(inverse_model_path),
                    "surrogate_dense_units": json.dumps(surrogate_hidden_units),
                    "inverse_dense_units": json.dumps(inverse_hidden_units),
                    "surrogate_optimizer": "Adam",
                    "surrogate_learning_rate": float(surrogate_hp.get("learning_rate")),
                    "surrogate_dropout_rate": float(surrogate_hp.get("dropout_rate")),
                    "surrogate_l2_reg": float(surrogate_hp.get("l2_reg")),
                    "surrogate_use_batchnorm": bool(surrogate_hp.get("use_batchnorm")),
                    "surrogate_reconstruction_loss": SURROGATE_TRAIN_LOSS,
                    "surrogate_tuner_trials": KERAS_TUNER_TRIALS_SURROGATE,
                    "surrogate_tuner_best_val_loss": surrogate_best_val_loss,
                    "surrogate_tuner_best_epoch": surrogate_best_step,
                    "surrogate_jit_compile": False,
                    "surrogate_batch_size": SURROGATE_TRAIN_BATCH_SIZE,
                    "surrogate_early_stopping_patience": SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
                    "inverse_optimizer": "Adam",
                    "inverse_learning_rate": float(inverse_hp.get("learning_rate")),
                    "inverse_n_layers": int(inverse_hp.get("n_layers")),
                    "inverse_dropout_rate": float(inverse_hp.get("dropout_rate")),
                    "inverse_l2_reg": float(inverse_hp.get("l2_reg")),
                    "inverse_use_batchnorm": bool(inverse_hp.get("use_batchnorm")),
                    "inverse_reconstruction_loss": TRAIN_LOSS,
                    "range_penalty_weight": float(inverse_hp.get("penalty_weight")),
                    "inverse_tuner_trials": KERAS_TUNER_TRIALS_INVERSE,
                    "inverse_tuner_best_val_loss": inverse_best_val_loss,
                    "inverse_tuner_best_epoch": inverse_best_step,
                    "inverse_jit_compile": False,
                    "inverse_batch_size": INVERSE_BATCH_SIZE,
                    "inverse_early_stopping_patience": INVERSE_EARLY_STOPPING_PATIENCE,
                    **{f"surrogate_train_{key}": value for key, value in surrogate_train_metrics.items()},
                    **{f"surrogate_val_{key}": value for key, value in surrogate_val_metrics.items()},
                    **{f"surrogate_test_{key}": value for key, value in surrogate_test_metrics.items()},
                    **{f"train_{key}": value for key, value in train_metrics.items()},
                    **{f"val_{key}": value for key, value in val_metrics.items()},
                    **{f"test_{key}": value for key, value in test_metrics.items()},
                    **{f"test_{key}": value for key, value in test_range_stats.items()},
                }
            )

            # Write incrementally so a long sweep stays recoverable if interrupted.
            pd.DataFrame(rows).to_csv(OUT_PATH, index=False)

            del inverse_model, combined_model
            tf.keras.backend.clear_session()
            gc.collect()

    out = pd.DataFrame(rows)
    out.to_csv(OUT_PATH, index=False)
    print()
    print(f"wrote {OUT_PATH}")
    print(f"saved sweep models to {SWEEP_MODEL_DIR}")
else:
    print(f"Skipping training. Reading existing results from {OUT_PATH}")
    out = pd.read_csv(OUT_PATH)

out.head()


Tuning a surrogate (50 trials) and inverse (50 trials) per fraction.
Tuner working directory: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/kt_dir2/uniform_data_amount_sweep

=== 10% of training pool (135 samples) ===
  [surrogate] tuning 50 trials...


I0000 00:00:1783384803.327728    1725 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1783384803.328281    1725 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38660 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 4g.40gb, pci bus id: 0000:05:00.0, compute capability: 8.0
I0000 00:00:1783384810.049873    1947 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/home/olivias/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


    best surrogate units=[32] lr=0.04168 val_loss=0.00754126
  [inverse] tuning 50 trials...


/home/olivias/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


    best inverse units=[192] lr=0.0901 penalty=0.01354 val_loss=0.0123969

=== 20% of training pool (270 samples) ===
  [surrogate] tuning 50 trials...


...

    best inverse units=[320] lr=0.03213 penalty=0.01442 val_loss=0.00352979

=== 80% of training pool (1082 samples) ===
  [surrogate] tuning 50 trials...


/home/olivias/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


    best surrogate units=[864] lr=0.03429 val_loss=0.00246742
  [inverse] tuning 50 trials...


/home/olivias/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


    best inverse units=[192] lr=0.01468 penalty=0.01602 val_loss=0.00322258

=== 90% of training pool (1217 samples) ===
  [surrogate] tuning 50 trials...


In [ ]:
summary = (
    out.groupby(["training_percent", "n_samples"], as_index=False)
    .agg(
        train_mean=("train_mean_hamiltonian_pct", "mean"),
        train_std=("train_mean_hamiltonian_pct", "std"),
        val_mean=("val_mean_hamiltonian_pct", "mean"),
        val_std=("val_mean_hamiltonian_pct", "std"),
        test_mean=("test_mean_hamiltonian_pct", "mean"),
        test_std=("test_mean_hamiltonian_pct", "std"),
    )
    .sort_values("training_percent")
)
summary.to_csv(SUMMARY_OUT_PATH, index=False)
print(f"wrote {SUMMARY_OUT_PATH}")
summary
